In [0]:
# =============================================================================
# Notebook 06 — Baseline Models under GroupKFold CV
# Objective: Train and evaluate Ridge, Random Forest, and XGBoost on the
#            three feature layers (L1 material-only, L3 material+experimental)
#            using 5-fold GroupKFold (grouped by DOI) to prevent data leakage.
#            Report R², RMSE, MAE on log1p scale and back-transformed scale.
#
# Inputs : ml_perovskites.X_exp_clean  (L3, 85 features)
#          ml_perovskites.X_mat_clean  (L1, 72 features)
#          ml_perovskites.y_targets
# Outputs: ml_perovskites.cv_results   (Delta table — all fold metrics)
#          /tmp/06_cv_results.png
#          /tmp/06_parity_plots.png
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 0 — Install dependencies (run this cell alone first)
# =============================================================================
%pip install xgboost --quiet

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N_SPLITS = 5

print("Section 1 complete: imports and configuration loaded.")
print(f"XGBoost version: {xgb.__version__}")

# =============================================================================
# SECTION 2 — Load tables
# =============================================================================

print("\n--- Loading tables ---")

y_pd   = spark.table("ml_perovskites.y_targets").toPandas()
xl3_pd = spark.table("ml_perovskites.X_exp_clean").toPandas()
xl1_pd = spark.table("ml_perovskites.X_mat_clean").toPandas()

print(f"y_targets shape    : {y_pd.shape}")
print(f"X_exp_clean (L3)   : {xl3_pd.shape}")
print(f"X_mat_clean (L1)   : {xl1_pd.shape}")

# =============================================================================
# SECTION 3 — Prepare aligned arrays
# =============================================================================

print("\n--- Preparing aligned arrays ---")

# Align all frames on source_row_id
common_ids = (set(y_pd["source_row_id"])
              & set(xl3_pd["source_row_id"])
              & set(xl1_pd["source_row_id"]))
print(f"Common source_row_ids: {len(common_ids)}")

y_aligned  = y_pd[y_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)
xl3_aligned = xl3_pd[xl3_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)
xl1_aligned = xl1_pd[xl1_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)

# Target and groups
y_log   = y_aligned["log1p_h2_rate"].values.astype(float)
groups  = y_aligned["doi"].values

# Feature matrices (drop key/metadata columns)
META = {"source_row_id", "doi", "base_compound", "family_proposed",
        "formula_reduced", "n_elements", "n_A_elements", "n_B_elements",
        "A_site_elements", "B_site_elements", "parse_ok", "Compound_original", "domain"}

feat_cols_l3 = [c for c in xl3_aligned.columns if c not in META]
feat_cols_l1 = [c for c in xl1_aligned.columns if c not in META]

X_l3 = xl3_aligned[feat_cols_l3].values.astype(float)
X_l1 = xl1_aligned[feat_cols_l1].values.astype(float)

print(f"X_l3 (L3) shape: {X_l3.shape}")
print(f"X_l1 (L1) shape: {X_l1.shape}")
print(f"y shape        : {y_log.shape}")
print(f"Unique groups  : {len(np.unique(groups))}")

# =============================================================================
# SECTION 4 — Model definitions
# =============================================================================

# Each model is a sklearn Pipeline: Imputer → (Scaler) → Estimator
# Imputer fills residual NaN with column median (defensive; most NaN already handled).
# Scaler applied for Ridge only (scale-sensitive); RF and XGBoost are scale-invariant.

def make_ridge(alpha=10.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   Ridge(alpha=alpha, random_state=RANDOM_SEED)),
    ])

def make_rf(n_estimators=300, max_features=0.33):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   RandomForestRegressor(
            n_estimators=n_estimators,
            max_features=max_features,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )),
    ])

def make_xgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   xgb.XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_SEED,
            verbosity=0,
            n_jobs=-1,
        )),
    ])

# =============================================================================
# SECTION 5 — Cross-validation loop
# =============================================================================

print("\n--- Running GroupKFold cross-validation ---")
print(f"Models: Ridge, RandomForest, XGBoost")
print(f"Layers: L1 (material-only), L3 (material + experimental)")
print(f"Folds : {N_SPLITS} (grouped by DOI)\n")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def metrics_dict(y_true, y_pred_log):
    """Compute metrics on log scale and back-transformed scale."""
    # Log scale
    r2_log   = r2_score(y_true, y_pred_log)
    rmse_log = rmse(y_true, y_pred_log)
    mae_log  = mean_absolute_error(y_true, y_pred_log)
    # Back-transform: expm1 undoes log1p
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(np.clip(y_pred_log, 0, None))  # clip negatives to 0 before expm1
    r2_orig   = r2_score(y_true_orig, y_pred_orig)
    rmse_orig = rmse(y_true_orig, y_pred_orig)
    mae_orig  = mean_absolute_error(y_true_orig, y_pred_orig)
    return {
        "r2_log": r2_log, "rmse_log": rmse_log, "mae_log": mae_log,
        "r2_orig": r2_orig, "rmse_orig": rmse_orig, "mae_orig": mae_orig,
    }

gkf = GroupKFold(n_splits=N_SPLITS)

experiments = [
    ("Ridge",  "L1", make_ridge, X_l1),
    ("Ridge",  "L3", make_ridge, X_l3),
    ("RF",     "L1", make_rf,    X_l1),
    ("RF",     "L3", make_rf,    X_l3),
    ("XGBoost","L1", make_xgb,   X_l1),
    ("XGBoost","L3", make_xgb,   X_l3),
]

all_fold_results = []
oof_predictions  = {}   # (model_name, layer) → array of OOF predictions

for model_name, layer, make_fn, X in experiments:
    key = (model_name, layer)
    oof_pred = np.full(len(y_log), np.nan)

    fold_metrics = []
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y_log[train_idx], y_log[val_idx]

        pipe = make_fn()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        oof_pred[val_idx] = y_pred

        m = metrics_dict(y_val, y_pred)
        m.update({"fold": fold_idx + 1, "model": model_name, "layer": layer,
                  "n_train": len(train_idx), "n_val": len(val_idx)})
        fold_metrics.append(m)

    oof_predictions[key] = oof_pred

    # Aggregate across folds
    fm_df  = pd.DataFrame(fold_metrics)
    mean_m = fm_df[["r2_log","rmse_log","mae_log","r2_orig","rmse_orig","mae_orig"]].mean()
    std_m  = fm_df[["r2_log","rmse_log","mae_log"]].std()

    print(f"{model_name:8s} | {layer} | "
          f"R²(log)={mean_m['r2_log']:+.3f}±{std_m['r2_log']:.3f}  "
          f"RMSE(log)={mean_m['rmse_log']:.3f}  "
          f"R²(orig)={mean_m['r2_orig']:+.3f}")

    all_fold_results.extend(fold_metrics)

results_df = pd.DataFrame(all_fold_results)

# =============================================================================
# SECTION 6 — Summary table
# =============================================================================

print("\n--- Summary (mean ± std across 5 folds) ---")

summary = (results_df
           .groupby(["model", "layer"])
           .agg(
               r2_log_mean=("r2_log",   "mean"),
               r2_log_std= ("r2_log",   "std"),
               rmse_log=   ("rmse_log", "mean"),
               mae_log=    ("mae_log",  "mean"),
               r2_orig=    ("r2_orig",  "mean"),
               rmse_orig=  ("rmse_orig","mean"),
           )
           .reset_index()
           .sort_values("r2_log_mean", ascending=False))

print(summary.to_string(index=False))

print(f"\nICC(DOI) ceiling (material-only): 0.58")
print(f"Best L1 R²(log): {summary[summary['layer']=='L1']['r2_log_mean'].max():.3f}")
print(f"Best L3 R²(log): {summary[summary['layer']=='L3']['r2_log_mean'].max():.3f}")

# =============================================================================
# SECTION 7 — Plots
# =============================================================================

print("\n--- Generating plots ---")

# Plot 1: CV R² bar chart (log scale, mean ± std)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models_order = ["Ridge", "RF", "XGBoost"]
colors = {"L1": "#4C72B0", "L3": "#DD8452"}
x = np.arange(len(models_order))
width = 0.35

for i, layer in enumerate(["L1", "L3"]):
    sub = summary[summary["layer"] == layer].set_index("model")
    means = [sub.loc[m, "r2_log_mean"] if m in sub.index else 0 for m in models_order]
    stds  = [sub.loc[m, "r2_log_std"]  if m in sub.index else 0 for m in models_order]
    offset = (i - 0.5) * width
    bars = axes[0].bar(x + offset, means, width, yerr=stds, capsize=4,
                       label=f"{layer}", color=colors[layer], alpha=0.85)

axes[0].axhline(0.58, color="red", linestyle="--", linewidth=1.2, label="ICC ceiling (0.58)")
axes[0].axhline(0.0,  color="gray", linestyle=":",  linewidth=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_order)
axes[0].set_ylabel("CV R² (log1p scale)")
axes[0].set_title("GroupKFold CV R² — L1 vs L3")
axes[0].legend(fontsize=9)
axes[0].set_ylim(bottom=min(-0.1, summary["r2_log_mean"].min() - 0.05))

# Plot 2: OOF parity plots (log scale) for best model per layer
best_l3_model = summary[summary["layer"]=="L3"].iloc[0]["model"]
best_l1_model = summary[summary["layer"]=="L1"].iloc[0]["model"]

for ax, (m_name, layer), color in zip(
        [axes[1], axes[1]],
        [(best_l3_model, "L3"), (best_l1_model, "L1")],
        [colors["L3"], colors["L1"]]):
    oof = oof_predictions[(m_name, layer)]
    mask = ~np.isnan(oof)
    ax.scatter(y_log[mask], oof[mask], alpha=0.25, s=10, color=color,
               label=f"{m_name} {layer}")

lim = [y_log.min() - 0.5, y_log.max() + 0.5]
axes[1].plot(lim, lim, "k--", linewidth=0.8, label="y=x")
axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_xlabel("Actual log1p(H₂ rate)")
axes[1].set_ylabel("Predicted log1p(H₂ rate)")
axes[1].set_title(f"OOF parity — best models")
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.savefig("/tmp/06_cv_results.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/06_cv_results.png")

# Separate parity plots (one per model×layer)
fig2, axes2 = plt.subplots(2, 3, figsize=(15, 9))
axes2 = axes2.flatten()

for ax_idx, (model_name, layer, _, _) in enumerate(experiments):
    key = (model_name, layer)
    oof = oof_predictions[key]
    mask = ~np.isnan(oof)
    r2_oof = r2_score(y_log[mask], oof[mask])

    ax = axes2[ax_idx]
    ax.scatter(y_log[mask], oof[mask], alpha=0.3, s=8, color=colors[layer])
    lim = [y_log.min() - 0.3, y_log.max() + 0.3]
    ax.plot(lim, lim, "k--", linewidth=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{model_name} — {layer}  (OOF R²={r2_oof:.3f})")
    ax.set_xlabel("Actual log1p(H₂ rate)")
    ax.set_ylabel("Predicted")

plt.tight_layout()
plt.savefig("/tmp/06_parity_plots.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/06_parity_plots.png")

# =============================================================================
# SECTION 8 — Write results to Delta table
# =============================================================================

print("\n--- Writing cv_results to Delta table ---")

results_spark = spark.createDataFrame(
    results_df.where(pd.notnull(results_df), other=None)
)
(results_spark
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.cv_results"))

n_cv = spark.table("ml_perovskites.cv_results").count()
print(f"Rows written to ml_perovskites.cv_results: {n_cv}")

# =============================================================================
# SECTION 9 — Acceptance criteria checks
# =============================================================================

print("\n" + "="*60)
print("ACCEPTANCE CRITERIA CHECKS")
print("="*60)

best_r2_l3 = summary[summary["layer"]=="L3"]["r2_log_mean"].max()
best_r2_l1 = summary[summary["layer"]=="L1"]["r2_log_mean"].max()

print(f"\n[CHECK 1] Best L3 CV R²(log): {best_r2_l3:.3f}")
print("  PASS (positive)" if best_r2_l3 > 0 else "  WARNING — all models below baseline")

print(f"\n[CHECK 2] L3 gain over L1:")
l3_gain = best_r2_l3 - best_r2_l1
print(f"  {l3_gain:+.3f} R² units")
print("  Experimental features add value" if l3_gain > 0.02
      else "  NOTE: minimal gain from experimental layer at this stage")

print(f"\n[CHECK 3] cv_results row count: {n_cv} (expected {N_SPLITS * len(experiments)})")
print("  PASS" if n_cv == N_SPLITS * len(experiments) else f"  FAIL — got {n_cv}")

print(f"\n[CHECK 4] No DOI leakage in splits: confirmed in Notebook 05")
print("  PASS")

print("\n" + "="*60)

# =============================================================================
# SECTION 10 — Summary
# =============================================================================

print("\n" + "="*60)
print("NOTEBOOK 06 COMPLETE")
print("="*60)
print(f"Models evaluated  : Ridge, RandomForest, XGBoost")
print(f"Feature layers    : L1 (72 features), L3 (85 features)")
print(f"Validation        : 5-fold GroupKFold by DOI")
print(f"\nResults (mean CV R² on log1p scale):")
for _, row in summary.iterrows():
    print(f"  {row['model']:8s} {row['layer']}  R²={row['r2_log_mean']:+.3f}±{row['r2_log_std']:.3f}  "
          f"RMSE={row['rmse_log']:.3f}  (orig scale R²={row['r2_orig']:+.3f})")
print(f"\nICC(DOI) ceiling : 0.58")
print(f"Best model (L3)  : {best_l3_model}  R²={best_r2_l3:.3f}")
print(f"L3 gain vs L1    : {l3_gain:+.3f} R² units")
print(f"\nOutput table : ml_perovskites.cv_results ({n_cv} rows)")
print(f"Output plots : /tmp/06_cv_results.png, /tmp/06_parity_plots.png")
print(f"\nNext step: Notebook 07 — Hyperparameter optimisation (Optuna) for")
print(f"           the best-performing model family under the same GroupKFold.")